# Methoden en Technieken 2025-2026 -- Blok 3

## Datapunt Opdracht 3a

In deze opdracht worden de volgende leeruitkomsten getoetst, relevante termen zijn **dik** gedrukt:
- A2: Je stelt voor een AI-oplossing juridische, ethische, organisatorische, **functionele en technische requirements** op.
- B1: Je **verkent en prepareert een dataset voor het trainen en testen van een AI-model en kan de voor- en nadelen van het gebruik van een bestaande dataset onderbouwen**, rekening houdend met technische en ethische randvoorwaarden.
- B2: Je **stelt op basis van requirements en data een geschikte architectuur voor een AI-oplossing op en selecteert daarvoor passende AI-technieken gebruik makend van bijvoorbeeld** **machine learning**, deep learning, kennisrepresentatie, computer vision en **natural language processing**.
- B3: Je **ontwikkelt een nieuw** of voorgetraind **AI-model volgens een iteratief en systematisch proces**.
- C2: **Je evalueert en beoordeelt de kwaliteit van een AI-model aan de hand van kwaliteitscriteria die in het vakgebied erkend worden** zoals robustness, **performance**, scalability, explainability, **model complexity** en resource demand.


## De opdracht

Onderstaande code leest de data van verschillende *ratings* in. Deze dataset is de **MovieTweetings**-dataset (ook naar verwezen in Les 4 van blok 3) waar het MovieGEEKs-voorbeeld gebruik van maakt. In de data staan de waarderingen (van 0 t/m 10) van gebruikers voor verschillende films en bijbehorende *timestamp*. Zie ook https://github.com/sidooms/MovieTweetings/tree/master voor een uitleg van de dataset.

De bedoeling is om een aanbevelings-systeem te bouwen dat voor elke willekeurige gebruiker in het systeem drie films aanbeveelt. Probeer de aanbeveling zo persoonlijk mogelijk te maken.
* Kies een model en verantwoord deze keuze.
* Besluit hoe je het model beoordeelt (datasplitsing en maatstaf) en verantwoord deze keuze.
* Evalueer het model.
* Geef concrete suggesties om het model te verbeteren. Je hoeft deze verbeteringen niet uit te voeren.
* Bespreek voor- en nadelen van het model dat je hebt gemaakt.

In [1]:
# %pip install pandas surprise numpy==1.26.4 scikit-learn

In [2]:
import os
# import torch

# Environment configuration
# os.environ["KERAS_BACKEND"] = "torch"

import pandas as pd
import numpy as np
import math
import warnings

from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import GridSearchCV as SurpriseGridSearchCV
from surprise.model_selection import KFold as SurpriseKFold
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')
np.random.seed(42)

# Hardware check
# if torch.cuda.is_available():
#     print('Notebook gebruikt windows GPU')
# elif torch.mps.is_available():
#     print('Notebook gebruikt apple GPU')
# else:
#     print('Notebook gebruikt CPU')


## Data laden

Onderstaande code leest de data van verschillende *ratings* in. Deze dataset is de **MovieTweetings**-dataset (ook naar verwezen in Les 4 van blok 3) waar het MovieGEEKs-voorbeeld gebruik van maakt. In de data staan de waarderingen (van 0 t/m 10) van gebruikers voor verschillende films en bijbehorende *timestamp*. Zie ook https://github.com/sidooms/MovieTweetings/tree/master voor een uitleg van de dataset.

In [3]:
ratings = pd.read_csv(
    'https://raw.githubusercontent.com/sidooms/MovieTweetings/master/latest/ratings.dat',
    delimiter='::', engine='python', header=None,
    names=['user_id', 'movie_id', 'rating', 'timestamp']
)
print(f"Loaded {len(ratings):,} ratings")

Loaded 921,398 ratings


### Movie metadata

Naast de ratings laad ik ook de film metadata (titel, jaar, genres) in. Deze wil ik gebruiken om een content-based component te maken voor een hybride model en om de uiteindelijke aanbevelingen beter interpreteerbaar te presenteren door genre informatie te kunnen zien.

In [4]:
items_raw = pd.read_csv(
    'https://raw.githubusercontent.com/sidooms/MovieTweetings/master/latest/movies.dat',
    delimiter='::', engine='python', header=None,
    names=['movie_id', 'title_raw', 'genres_raw'],
    encoding='utf-8'
)

# Haal het jaar uit de titel, bijv. "Toy Story (1995)" → title="Toy Story", year=1995
items_raw['year'] = items_raw['title_raw'].str.extract(r'\((\d{4})\)').astype(float)
items_raw['title'] = items_raw['title_raw'].str.replace(r'\s*\(\d{4}\)\s*$', '', regex=True)

# Splits genres op in een lijst en vang lege waarden op
items_raw['genres'] = (
    items_raw['genres_raw'].fillna('').str.split('|', regex=False)
    .apply(lambda g: [x for x in g if x])
)

# Zorg voor unieke film-metadata per movie_id
items = (
    items_raw[['movie_id', 'title', 'year', 'genres']]
    .drop_duplicates(subset=['movie_id'], keep='last')
    .reset_index(drop=True)
)
print(f"Loaded {len(items):,} unique movies")

# Combineer ratings met filinformatie
df = ratings.merge(items, on='movie_id', how='inner')

# Zorg dat rating numeriek is en binnen 0–10 valt
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df = df[(df['rating'] >= 0) & (df['rating'] <= 10)].dropna(subset=['rating', 'genres'])

Loaded 38,013 unique movies


## Eerste inspectie van de data

Een snelle controle op het aantal observaties, kolomtypen en ontbrekende waarden vóór verdere verkenning. Gedetailleerde exploratie volgt na de datasplit, uitsluitend op de trainingsset.

In [5]:
print(f"Aantal observaties: {len(df):,}")
print(f"Aantal kolommen:    {df.shape[1]}")
print(f"Unieke gebruikers:  {df['user_id'].nunique():,}")
print(f"Unieke films:       {df['movie_id'].nunique():,}")
print()
df.info()

Aantal observaties: 921,398
Aantal kolommen:    7
Unieke gebruikers:  71,707
Unieke films:       38,013

<class 'pandas.DataFrame'>
RangeIndex: 921398 entries, 0 to 921397
Data columns (total 7 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   user_id    921398 non-null  int64  
 1   movie_id   921398 non-null  int64  
 2   rating     921398 non-null  int64  
 3   timestamp  921398 non-null  int64  
 4   title      921398 non-null  str    
 5   year       921398 non-null  float64
 6   genres     921398 non-null  object 
dtypes: float64(1), int64(4), object(1), str(1)
memory usage: 49.2+ MB


## Voor- en nadelen van het gebruik van de dataset

### Voordelen

- De dataset is publiek beschikbaar en open voor onderzoek, waardoor experimenten reproduceerbaar zijn en resultaten eenvoudig met andere studies kunnen worden vergeleken.
- De dataset wordt veel gebruikt in recommender system onderzoek, waardoor prestaties van modellen goed te vergelijken zijn met eerder werk.
- De ratings zijn afkomstig van echte gebruikersinteracties op sociale media en zijn dus niet kunstmatig gegenereerd.
- De datastructuur is eenvoudig en vergelijkbaar met bekende datasets zoals MovieLens (users, items en ratings), waardoor deze makkelijk te importeren en te gebruiken is.
- De ratings gebruiken een schaal van 0 tot10, waardoor meer nuance mogelijk is dan bij bijvoorbeeld een 5-sterren systeem.
- De dataset bevat timestamps voor elke rating waardoor tijdsgebaseerde analyses of temporele train/test splits mogelijk zijn.
- Films worden geïdentificeerd met IMDb identifiers, waardoor eenvoudig extra metadata kan worden toegevoegd uit externe bronnen.
- De dataset bevat basis metadata zoals filmgenres, waardoor ook content-based of hybride recommender systemen onderzocht kunnen worden

### Nadelen

- De dataset wordt niet meer actief geüpdatet en bevat sinds 2021 geen nieuwe ratings meer, waardoor deze minder representatief is voor recente films en gebruikersvoorkeuren.
- De dataset is sterk sparse, omdat veel gebruikers slechts een klein aantal films beoordelen en veel films weinig ratings hebben.
- Door de sparsity ontstaan veel cold-start gebruikers en films, wat het moeilijker maakt om betrouwbare aanbevelingen te genereren.
- De dataset bevat alleen ratings van gebruikers die hun IMDb beoordeling via Twitter delen, waardoor de gebruikerspopulatie minder representatief is voor filmkijkers in het algemeen.
- De dataset bevat weinig gebruikersmetadata zoals leeftijd, locatie of geslacht, wat recommendations op basis van user similarity moeilijk maakt.
- De inhoudelijke metadata van films is beperkt tot basisinformatie zoals titel en genre, waardoor extra data nodig kan zijn voor uitgebreidere content-based modellen.
- De ratings worden automatisch uit tweets geëxtraheerd, waardoor het mogelijk is dat sommige ratings verkeerd worden geïnterpreteerd.
- De dataset bevat Twitter user IDs, wat als persoonlijke data kan worden beschouwd omdat je user names er mee kan opvragen.
- Omdat de dataset bekend is en veel gebruikt wordt in onderzoek, kan voorafgaande kennis over eigenschappen zoals sparsity of ratingdistributies een vorm van data leakage veroorzaken tijdens exploratie.
- De dataset heeft een populariteitsbias, omdat films waar vaker over getweet wordt waarschijnlijk ook vaker ratings krijgen.

## Data Preparation

### Datasetopschoning

Voordat het recommender model wordt getraind wordt de dataset eerst opgeschoond. Gebruikers met zeer weinig interacties worden gefilterd omdat zij onvoldoende informatie bieden om betrouwbare aanbevelingen te genereren [1] *(H9, p.229: “You'll usually filter users away with only a small number of interactions in your data”)*.  

Datasets voor recommender systemen vereisen doorgaans enige opschoning, omdat gebruikers met slechts enkele ratings weinig bijdragen aan het leerproces van het model en de evaluatie kunnen verstoren [1] *(H9, p.236: “Most data sets need a bit of housekeeping before you evaluate them… all the users who rated only a few items don't help the recommender much”)*.

In dit project worden daarom gebruikers met minder dan vijf ratings verwijderd de dataset is al bekend wat enige data leakage is maar hierdoor is al bekend dat er veel users met minder dan 5 ratings zijn waar niet goed voor gepredict kan worden op basis van voorkeur. Daarnaast worden films met minder dan drie ratings gefilterd, omdat zeer zeldzame films geïsoleerde kolommen in de gebruikers–item matrix creëren en de sparsity vergroten.

### Minimum ratings voor collaborative filtering

Collaborative filtering vereist voldoende interactiedata om gelijkenissen tussen gebruikers of items te berekenen. Gebruikers met slechts één rating leveren daarom geen bruikbare informatie voor dit type algoritme [1] *(H9, p229: “The algorithm won't work for users with only one rating”)*. Door een minimum aantal ratings te vereisen ontstaat meer overlap tussen gebruikersinteracties, wat essentieel is voor betrouwbare similarityberekeningen [1] p.242.

### Chronologische ordening en datasplit

Per gebruiker worden ratings chronologisch gesorteerd op timestamp. Vervolgens wordt een given-n protocol toegepast: de eerste 5 ratings gaan naar de trainingsset, de eerste helft van de resterende ratings naar de validatieset, en de tweede helft naar de testset.

De chronologische ordening heeft als voordeel dat het evaluatieproces realistischer is. In echte aanbevelingssystemen worden aanbevelingen ook gebaseerd op interacties uit het verleden om toekomstig gedrag te voorspellen. Daarnaast kan de smaak van gebruikers in de loop van de tijd veranderen, waardoor een tijdsgebaseerde split beter weerspiegelt hoe voorkeuren veranderen. Hierdoor wordt voorkomen dat informatie uit de “toekomst” onbedoeld in het trainingsproces terechtkomt en wordt deze dataleakage vermeden.

In [6]:
# Verwijder gebruikers met minder dan 5 ratings
user_counts = df.groupby('user_id').size()
valid_users = user_counts[user_counts >= 5].index
df = df[df['user_id'].isin(valid_users)].copy()

print(f"Na filteren gebruikers (< 5 ratings): {len(df):,} ratings  ({df['user_id'].nunique():,} users)")

# Verwijder films met minder dan 3 ratings
movie_counts = df.groupby('movie_id').size()
valid_movies = movie_counts[movie_counts >= 3].index
df = df[df['movie_id'].isin(valid_movies)].copy()

print(f"Na filteren films (< 3 ratings):      {len(df):,} ratings  ({df['user_id'].nunique():,} users, {df['movie_id'].nunique():,} movies)")

Na filteren gebruikers (< 5 ratings): 845,154 ratings  (23,805 users)
Na filteren films (< 3 ratings):      819,491 ratings  (23,802 users, 16,384 movies)


## Data Split met Given-n Protocol (chronologisch)

### Train / Validatie / Test datasplitsing

Er wordt een drieledige datasplitsing toegepast:

- Trainset: Wordt gebruikt om het model te trainen. Deze bevat het eerste N aantal ratings per gebruiker.

- Validatieset: Wordt gebruikt voor model ontwikkeling en het afstellen van hyperparameters. Deze bevat de eerste helft van de resterende ratings.

- Testset: Wordt uitsluitend gebruikt voor de finale evaluatie van het model. Deze bevat de tweede helft van de resterende ratings.

De splitsing gebeurt per gebruiker. De vroegste interacties van een gebruiker worden in de trainingsset geplaatst, terwijl latere interacties naar validatie en test gaan. Hierdoor komt elke gebruiker voor in de trainingsdata en wordt de evaluatie realistischer. Dit is tevens veel voorkoment in papers. [1] *(H9.5, p234: “Dividing the data like this is called the given n
protocol; it’s mentioned here because it’s often used in research papers”)*.

### Given-n protocol

In dit project wordt een given-n protocol gebruikt met (n = 5). Voor elke gebruiker worden alle ratings chronologisch gesorteerd. De eerste vijf ratings worden in de trainingsset geplaatst. Van de resterende ratings gaat de eerste helft naar de validatieset en de tweede helft naar de testset.

### Cold-start gebruikers

Gebruikers met minder dan vijf ratings in de trainingsset worden als cold-start beschouwd en ontvangen aanbevelingen via een populariteitsbaseline.

In [7]:
GIVEN_N = 5

def given_n_split(data, n=GIVEN_N):
    """
    Chronologische train / validatie / test split per gebruiker
    """
    train_parts, val_parts, test_parts = [], [], []
    for _, group in data.groupby('user_id'):
        sorted_group = group.sort_values('timestamp')
        train_parts.append(sorted_group.iloc[:n])
        remaining = sorted_group.iloc[n:]
        if len(remaining) > 0:
            mid = len(remaining) // 2

            # oneven is extra rating naar validatie
            mid = max(mid, 1)
            val_parts.append(remaining.iloc[:mid])
            if len(remaining) > mid:
                test_parts.append(remaining.iloc[mid:])

    train = pd.concat(train_parts).reset_index(drop=True)
    val = pd.concat(val_parts).reset_index(drop=True) if val_parts else pd.DataFrame(columns=data.columns)
    test = pd.concat(test_parts).reset_index(drop=True) if test_parts else pd.DataFrame(columns=data.columns)
    return train, val, test

train_df, val_df, test_df = given_n_split(df, n=GIVEN_N)

#  elke val/test-gebruiker ook in de trainingsdata
train_user_set = set(train_df['user_id'].unique())
val_df = val_df[val_df['user_id'].isin(train_user_set)].copy()
test_df = test_df[test_df['user_id'].isin(train_user_set)].copy()

train_val_df = pd.concat([train_df, val_df]).reset_index(drop=True)

print(f"Train:     {len(train_df):>8,} ratings  ({train_df['user_id'].nunique():,} users)")
print(f"Val:       {len(val_df):>8,} ratings  ({val_df['user_id'].nunique():,} users)")
print(f"Test:      {len(test_df):>8,} ratings  ({test_df['user_id'].nunique():,} users)")
print(f"Train+Val: {len(train_val_df):>8,} ratings  ({train_val_df['user_id'].nunique():,} users)")
print(f"\nAlle val-gebruikers in training:  {set(val_df['user_id'].unique()).issubset(train_user_set)}")
print(f"Alle test-gebruikers in training: {set(test_df['user_id'].unique()).issubset(train_user_set)}")

# Identificeer cold-start gebruikers (< 5 training ratings)
COLD_THRESHOLD = 5
ratings_per_user = train_df.groupby('user_id').size().to_dict()
cold_users = {u for u, c in ratings_per_user.items() if c < COLD_THRESHOLD}
warm_users = {u for u, c in ratings_per_user.items() if c >= COLD_THRESHOLD}

print(f"\nCold-start gebruikers (< {COLD_THRESHOLD} training ratings): {len(cold_users):,}")
print(f"Warm gebruikers (>= {COLD_THRESHOLD} training ratings):      {len(warm_users):,}")

Train:      118,717 ratings  (23,802 users)
Val:        346,858 ratings  (21,205 users)
Test:       353,916 ratings  (19,185 users)
Train+Val:  465,575 ratings  (23,802 users)

Alle val-gebruikers in training:  True
Alle test-gebruikers in training: True

Cold-start gebruikers (< 5 training ratings): 228
Warm gebruikers (>= 5 training ratings):      23,574


## Data Exploration

In [8]:
# Basisstatistieken trainingsset
n_users_train = train_df['user_id'].nunique()
n_movies_train = train_df['movie_id'].nunique()
n_ratings_train = len(train_df)

print(f"Trainingsset overzicht:")
print(f"  Gebruikers: {n_users_train:,}")
print(f"  Films:      {n_movies_train:,}")
print(f"  Ratings:    {n_ratings_train:,}")
print(f"  Sparsity:   {1 - n_ratings_train / (n_users_train * n_movies_train):.4%}")
print(f"\nRating range: {train_df['rating'].min():.0f}–{train_df['rating'].max():.0f}")
print(f"Mean rating:  {train_df['rating'].mean():.2f}")

# Ratingdistributie
print("\nRatingdistributie (train):")
print(train_df['rating'].value_counts().sort_index().to_string())

# Genre-analyse
unique_genres = set()
for genres in train_df['genres'].dropna():
    if isinstance(genres, list):
        unique_genres.update(genres)
unique_genres = sorted(unique_genres)
print(f"\nUnieke genres: {len(unique_genres)}")

train_df.head(10)

Trainingsset overzicht:
  Gebruikers: 23,802
  Films:      10,239
  Ratings:    118,717
  Sparsity:   99.9513%

Rating range: 0–10
Mean rating:  7.61

Ratingdistributie (train):
rating
0        46
1      1462
2      1004
3      1629
4      2810
5      6934
6     12168
7     23417
8     29610
9     20484
10    19153

Unieke genres: 24


,user_id,movie_id,rating,timestamp,title,year,genres
0,3,10039344,5,1578603053,Countdown,2019.0,"[Horror, Thriller]"
1,3,6751668,9,1578955697,Gisaengchung,2019.0,[Drama]
2,3,358273,9,1579057827,Walk the Line,2005.0,"[Biography, Drama, Music, Romance]"
3,3,8579674,10,1579261830,1917,2019.0,"[Drama, War]"
4,3,7131622,8,1579559244,Once Upon a Time ...in Hollywood,2019.0,"[Comedy, Drama]"
5,4,2278871,8,1383419733,La vie d'Adèle,2013.0,"[Drama, Romance]"
6,4,2395417,8,1388170007,Still Life,2013.0,[Drama]
7,4,1800241,7,1388955438,American Hustle,2013.0,"[Crime, Drama]"
8,4,790636,8,1391207279,Dallas Buyers Club,2013.0,"[Biography, Drama]"
9,4,3344922,8,1422652427,Hungry Hearts,2014.0,"[Drama, Romance, Thriller]"


### Functionele en technische requirements

1. Het systeem genereert voor elke gebruiker drie gepersonaliseerde filmaanbevelingen, gerangschikt op voorspelde voorkeursscore. De output bevat precies 3 unieke, niet eerder beoordeelde films per gebruiker.

2. De architectuur is hybride: collaborative filtering en content-based filtering worden gecombineerd tot één aanbevelingsscore.

3. Het collaborative filtering component gebruikt matrixfactorisatie via SVD (Surprise-bibliotheek) om latente gebruikers- en itemfactoren te leren.

4. Het SVD-model modelleert gebruikers- en itembias zodat systematische verschillen in beoordelingsgedrag automatisch worden gecorrigeerd.

5. Filmgenres worden gerepresenteerd als one-hot vectors en overeenkomsten worden berekend met cosine similarity tussen gebruikersprofielen en filmvectoren.

6. Het gebruikersprofiel wordt opgebouwd uit het gemiddelde van de genrevectoren van films met rating ≥ 7.

7. De aanbevelingsscore wordt berekend als: `final_score = 0.8 × collaborative_score + 0.2 × content_score`.

8. Cold-start afhandeling: gebruikers met minder dan 5 training ratings krijgen aanbevelingen via een populariteitsbaseline met expliciete genre diversiteit. Films met minder dan 5 training ratings krijgen als collaborative score het gebruikersgemiddelde.

9. De eerste twe aanbevelingen worden gekozen op hoogste hybride score; de derde komt uit een ander genrecluster voor diversiteit (serindipity).

10. Het model behaalt een lagere RMSE en*hogere Precision@3 op de testset dan de populariteitsbaseline.

11. Hyperparameters worden systematisch geoptimaliseer* via grid search met k-fold CV op de trainingsdata. Het finale model wordt eenmalig op de testset geëvalueerd.

## Architectuur, hyperparameters en evaluatie

### Modelarchitectuur - hybride recommender

Voor hett recommender system ga ik een hybride systeem met collaborative filtering (CF) combineert met content-based filtering (CBF) gebruiken. Het CF-component is een SVD-model (matrixfactorisatie) dat latente voorkeuren leert uit de user–item ratingmatrix. Het CBF-component berekent genre cosine similarity tussen een gebruikersprofiel en filmgenrevectoren. Beide scores worden geïntegreerd via een gewogen lineaire combinatie: `final = 0.8 × collab + 0.2 × content`.

*& Hybride ipv van puur CF of puur CBF?**  
Puur CF faalt wanneer een gebruiker of film weinig ratings heeft (cold-start), terwijl CBF dan alsnog relevante aanbevelingen kan doen op basis van filmkenmerken. Puur CBF mist daarentegen het vermogen om onverwachte maar relevante films te ontdekken (serendipity), omdat het alleen films aanbeveelt die inhoudelijk lijken op wat de gebruiker al kent. Een hybride model benut beide signalen en is robuuster bij sparse data [1]. Doordat de dataset een bekende set is is ook al bekend dat de dataset sparse is. Daarnaast wordt in de litearuur expliciet beschreven dat hybride systemen beperkingen van individuele methoden kunnen verminderen [3] (Hybrid Methods: “Several recommendation systems use a hybrid approach by combining collaborative and content-based methods, which helps to avoid certain limitations of content-based and collaborative systems.”).

Een praktische implementatie van hybride systemen is het combineren van meerdere aanbevelingsscores tot eenbn eindscore [3] (Combining Separate Recommenders: “We can combine the outputs (ratings) obtained from individual recommender systems into one final recommendation using either a linear combination of ratings.”). Dit ondersteunt direct de gekozen integratiestrategie: `final = 0.8 × collab + 0.2 × content`.

**SVD ipv een ander CF-algoritme (bijv. KNN of ALS)?**  
SVD is gebaseerd op matrixfactorisatie, waarbij gebruikers en items worden gerepresenteerd in een latente factorruimte die interacties tussen gebruikersvoorkeuren en itemeigenschappen modelt [2] (Netflix Prize Competition: “Matrix factorization techniques have become a dominant methodology within collaborative filtering recommenders... they deliver accuracy superior to classical nearest-neighbor techniques...” ). Daarnaast modelleren veel matrixfactorisatiemodellen gebruikers en itembias, waardoor systematische ratingverschillen automatisch worden gecorrigeerd. In recommender data blijkt namelijk dat een groot deel van de variatie in ratings wordt veroorzaakt door systematische verschillen tussen gebruikers en items [2] (Adding Biases: “...much of the observed variation in rating values is due to effects associated with either users or items, known as biases...” ).

Een bijkomend voordeel van matrixfactorisatie is dat het goed kan omgaan met de hoge sparsity van recommender datasets. In praktijk bevatten ratingmatrices meestal slechts een klein deel van alle mogelijke user–item interacties [3] (Sparsity: “In any recommender system, the number of ratings already obtained is usually very small compared to the number of ratings that need to be predicted.”).

De Surprise-bibliotheek biedt ook een geoptimaliseerde SVD-implementatie met ingebouwde grid search, wat gebruikt kan worden om de juiste parameters te vinden.

**Cosine similarity voor content ipv Euclidische afstand of Jaccard?**  
Cosine similarity meet de hoek tussen vectoren en is daardoor ongevoelig voor de lengte van de genrevector dit is belangrijk omdat films verschillen in het aantal genres. In content-based recommender systemen wordt cosine similarity ook vaak gebruikt als standaardmaat voor vergelijkingen tussen item en gebruikersprofielen [3] (Content-Based Methods: “Utility function u(c,s) is usually represented … by some scoring heuristic defined in terms of vectors … such as the cosine similarity measure.”). Jaccard werkt alleen op sets en houdt geen rekening met het gebruikersprofiel als continuen vector (het gemiddelde van de genre vector).

### Hyperparameters

De SVD-hyperparameters worden geoptimaliseerd via grid search met 3-fold cross-validatio op de trainingsdata. De zoekruimte omvat:

- `n_factors` ∈ {50, 100, 150} — dimensionaliteit van de latente factorruimte
- `n_epochs` ∈ {20, 30} — aantal SGD-iteraties
- `lr_all` ∈ {0.002, 0.005, 0.01} — leersnelheid voor alle parameters
- `reg_all` ∈ {0.02, 0.05, 0.1} — L2-regularisatiesterkte

**Grid search keuze**  
Voor deze specifieke opdracht is de zoekruimte  klein (54 combinaties × 3 folds = 162 fits), waardoor grote grid search uit te voeren is en het volledige landschap wordt verkend. Voor een groter probleem is het grid searchen niet realistisch.

De beste parameters uit cross-validation worden gevalideerd op een aparte validatieset. Het utieindelijke model wordt daarna getraind op train + validatie samen, zodat het maximale datagebruik heeft voor de laatste testevaluatie.

### Evaluatiemaatregelen

De kwaliteit wordt gemeten twee metrieken die goed samen gaan volgens het boek [1].

- **RMSE (Root Mean Squared Error)** meet de afwijking tussen voorspelde en werkelijke ratings, waarbij grote fouten zwaarder worden bestraft. Dit is een veelgebruikte metriek voor rating prediction in recommender systemen en wordt vaak toegepast bij het evalueren van de nauwkeurigheid van voorspelde ratings. Het gebruik van RMSE voor evaluatie van aanbevelingsalgoritmen wordt ook beschreven in de literatuur [4] (Prediction Accuracy, p.273: “Root Mean Squared Error (RMSE) is perhaps the most popular metric used in evaluating accuracy of predicted ratings.”).

- **Precision@3** meet het aandeel relevante films (rating ≥ 7) in de top-3 aanbevelingen. Dit meet direct het doel van het systeem: relevante films bovenaan de aanbevelingslijst plaatsen. Ranking-gebaseerde evaluatiemetrics zoals precision worden vaak gebruikt wanneer het aantal aanbevelingen dat aan de gebruiker wordt getoond vooraf vaststaat. Dit wordt ook beschreven in de literatuur [4] (Measuring Usage Prediction, p.275: “In applications where the number of recommendations that can be presented to the user is preordained, the most useful measure of interest is Precision at N.”).

**Waarom niet MAE, Recall@K of NDCG?**  
MAE behandelt alle fouten gelijk en is minder geschikt wanneer grote voorspelfouten zwaarder moeten wegen. Recall@K is minder informatief bij K=3 en een zeer grote itemset, omdat de totale set relevante films per gebruiker groot kan zijn en recall daardoor structureel laag uitvalt. NDCG houdt rekening met de volgorde van aanbevelingen, maar bij slechts drie items is het rangordeverschil minimaal en voegt het weinig toe boven Precision@3.

### Validatiestrategie

Hyperparameters worden geoptimaliseerd via 3-fold cross-validation op de trainset om de optimale SVD-parameters te selecteren. De gevonden parameters worden daarna gevalideerd op de validatieset om te controleren dat de cross-validation resultaten te checken.  

De uiteindelijke modelprestaties worden dan gemeten op de testset die het model nooit eerder heeft gezien. Er wordt maar 1 iteratie gedaan. In dit project gebeurt de evaluatie via een offline experiment op een vooraf verzamelde dataset. Het gebruik van offline evaluatie voor recommender systemen wordt ook beschreven in de literatuur [4] (Offline Experiments, p.261: “An offline experiment is performed by using a pre-collected data set of users choosing or rating items. Using this data set we can try to simulate the behavior of users that interact with a recommendation system.”).

### Baseline-vergelijking

Het hybride model wordt vergeleken met een populariteitsbaseline:

waarbij de gemiddelde rating en n het aantal ratings is. Deze baseline beveelt dezelfde populaire films aan aan iedereen, waardoor objectief kan worden vastgesteld of personalisatie waarde toevoegt ten opzichte van een niet-gepersonaliseerde aanpak. De keuze van evaluatiemetrics en vergelijkingsmethoden hangt af van de eigenschappen die relevant zijn voor de specifieke toepassing. Dit wordt ook beschreven in de literatuur [4] (Recommendation System Properties, p.272: “As different applications have different needs, the designer of the system must decide on the important properties to measure for the concrete application at hand.”).

### Cold start strategy

Voor nieuwe gebruikers ontbreken interacties, waardoor voorkeuren nog niet betrouwbaar kunnen worden geschat. Daarom worden eerst populaire films aanbevolen om de kans op herkenning en interactie te vergroten. Tegelijk wordt genrediversiteit afgedwongen, zodat de eerste interacties informatieve voorkeurssignalen opleveren. In recommender systems wordt diversiteit gedefinieerd als het verminderen van similariteit tussen aanbevolen items, bijvoorbeeld door items uit verschillende categorieën of genres te selecteren [5] (“Diversity is generally discussed from the item side… it is defined based on the redundancy or similarity among the recommended items.”). Bij films zijn items uit hetzelfde genre doorgaans sterker aan elkaar gerelateerd dan films uit verschillende genres [5] (“Two movies both from romance will generally be more similar compared to two movies from romance and horror genre, respectively.”). Door populaire films met verschillende genres te tonen kan het systeem sneller relevante gebruikersvoorkeuren afleiden. Dit kan geimplementeerd worden door de meest populaire films aan te bevelen maar te filteren op genres zonder overlap.

## Base Model: Popularity Recommender

Een populariteitsbaseline dient als referentiepunt [1] *(H9.10.1, p239: Before you evaluate your new recommender, you should evaluate it on a simple recommender that, for example, always recommends the most popular items and see what numbers come out.)*

In [9]:
# Popularity baseline: score = gemiddelde rating × log
movie_stats = train_val_df.groupby('movie_id').agg(
    num_ratings=('rating', 'count'),
    avg_rating=('rating', 'mean')
).reset_index()

movie_stats['popularity_score'] = (
    movie_stats['avg_rating'] * np.log1p(movie_stats['num_ratings'])
)
movie_stats = movie_stats.merge(
    items[['movie_id', 'title', 'genres']], on='movie_id', how='left'
).sort_values('popularity_score', ascending=False).reset_index(drop=True)

# Snelle lookup-dicts voor later gebruik
popularity_dict = movie_stats.set_index('movie_id')['popularity_score'].to_dict()
movie_avg_dict = movie_stats.set_index('movie_id')['avg_rating'].to_dict()

print("Top-10 populairste films (baseline):\n")
print(
    movie_stats[['title', 'avg_rating', 'num_ratings', 'popularity_score']]
    .head(10)
    .to_string(index=False)
)

Top-10 populairste films (baseline):

                   title  avg_rating  num_ratings  popularity_score
            Interstellar    8.914506         1427         64.755240
                 Gravity    8.234586         1995         62.573803
 The Wolf of Wall Street    8.363902         1712         62.277626
The Shawshank Redemption    9.337571          708         61.290465
                    1917    8.526846         1192         60.406105
        Django Unchained    8.503923         1147         59.916743
        Captain Phillips    8.222686         1437         59.787221
                Whiplash    8.625125         1003         59.614682
        12 Years a Slave    8.346122         1199         59.174645
               Prisoners    8.149117         1415         59.126659


## Content-Based Filtering (genre similarity)

- One-hot encode de genres naar een binaire featurematrix.
- Bouw een gebruikersprofiel door de genrevectoren van films met rating ≥ 7 te middelen.
- Bereken cosine similarity tussen elk gebruikersprofiel en elke filmgenrevector.

In [10]:
# One-hot encode genres naar een binaire film × genre matrix
genre_encoder = MultiLabelBinarizer()
genre_matrix = pd.DataFrame(
    genre_encoder.fit_transform(items['genres']),
    columns=genre_encoder.classes_,
    index=items['movie_id']
)
genre_matrix = genre_matrix.groupby(level=0).max()

if '(no genres listed)' in genre_matrix.columns:
    genre_matrix.drop(columns=['(no genres listed)'], inplace=True)

all_genre_names = list(genre_matrix.columns)
print(f"Genre features ({len(all_genre_names)}): {', '.join(all_genre_names)}")

# Bouw gebruikersprofielen op basis van train+val (alle niet-testdata)
LIKE_THRESHOLD = 7
user_profiles = {}
for user_id, group in train_val_df.groupby('user_id'):
    liked_movies = group[group['rating'] >= LIKE_THRESHOLD]['movie_id']
    genre_vectors = genre_matrix.loc[genre_matrix.index.intersection(liked_movies)]

    # Fallback gebruik alle beoordeelde films als er geen 'liked' films zijn
    if len(genre_vectors) == 0:
        genre_vectors = genre_matrix.loc[genre_matrix.index.intersection(group['movie_id'])]

    if len(genre_vectors) > 0:
        user_profiles[user_id] = genre_vectors.mean().values
    else:
        user_profiles[user_id] = np.zeros(genre_matrix.shape[1])

# Bereken cosine similarity matrix (gebruikers × films)
user_ids_ordered = list(user_profiles.keys())
user_profile_matrix = np.array([user_profiles[u] for u in user_ids_ordered])
movie_ids_ordered = genre_matrix.index.tolist()

content_similarity = cosine_similarity(user_profile_matrix, genre_matrix.values)

user_to_index = {u: i for i, u in enumerate(user_ids_ordered)}
movie_to_index = {m: j for j, m in enumerate(movie_ids_ordered)}

def get_content_score(user_id, movie_id):
    """Geeft de cosine similarity tussen een gebruikersprofiel en een film."""
    ui = user_to_index.get(user_id)
    mi = movie_to_index.get(movie_id)
    if ui is None or mi is None:
        return 0.0
    return float(content_similarity[ui, mi])

print(f"Gebruikersprofielen gebouwd voor {len(user_profiles):,} users (op basis van train+val)")
print(f"Content-similarity matrix: {content_similarity.shape}")

Genre features (28): Action, Adult, Adventure, Animation, Biography, Comedy, Crime, Documentary, Drama, Family, Fantasy, Film-Noir, Game-Show, History, Horror, Music, Musical, Mystery, News, Reality-TV, Romance, Sci-Fi, Short, Sport, Talk-Show, Thriller, War, Western
Gebruikersprofielen gebouwd voor 23,802 users (op basis van train+val)
Content-similarity matrix: (23802, 38013)


## Collaborative Filtering — SVD met Hyperparameter Tuning

Surprise's SVD heeft een ingebouwde functie waardoor de biases verschillen opvangen in beoordelingsgedrag, daardoor is geen extra centering nodig.


In [11]:
# Gemiddelde rating per gebruiker (fallback voor onbekende films)
user_mean_rating = train_df.groupby('user_id')['rating'].mean().to_dict()

# Surprise dataset opbouwen (alleen trainingsdata voor hyper param)
reader = Reader(rating_scale=(0, 10))
surprise_data = Dataset.load_from_df(
    train_df[['user_id', 'movie_id', 'rating']], reader
)

# Grid search voor SVD hyperparameters (3-fold CV op trainingsdata)
param_grid = {
    'n_factors': [50, 100, 150],
    'n_epochs':  [20, 30],
    'lr_all':    [0.002, 0.005, 0.01],
    'reg_all':   [0.02, 0.05, 0.1],
}

gs = SurpriseGridSearchCV(SVD, param_grid, measures=['rmse'], cv=3,
                          refit=False, n_jobs=-1)
gs.fit(surprise_data)

best_params = gs.best_params['rmse']
print(f"\nBeste RMSE (CV): {gs.best_score['rmse']:.4f}")
print(f"Beste parameters: {best_params}")

# Validatiestap: train SVD op volledige trainset, evalueer op val_df
trainset_only = surprise_data.build_full_trainset()
svd_val = SVD(
    n_factors=best_params['n_factors'],
    n_epochs=best_params['n_epochs'],
    lr_all=best_params['lr_all'],
    reg_all=best_params['reg_all'],
    random_state=42
)
svd_val.fit(trainset_only)
val_predictions = [svd_val.predict(row['user_id'], row['movie_id'], row['rating'])
                   for _, row in val_df.iterrows()]
val_rmse = accuracy.rmse(val_predictions, verbose=False)
print(f"\nValidatie RMSE (SVD op val_df): {val_rmse:.4f}")

# final model: train op train+val samen met de beste parameters
surprise_data_final = Dataset.load_from_df(
    train_val_df[['user_id', 'movie_id', 'rating']], reader
)
trainset = surprise_data_final.build_full_trainset()

svd = SVD(
    n_factors=best_params['n_factors'],
    n_epochs=best_params['n_epochs'],
    lr_all=best_params['lr_all'],
    reg_all=best_params['reg_all'],
    random_state=42
)
svd.fit(trainset)

print(f"\nFinal SVD model getraind op train+val — factors={svd.n_factors}, epochs={svd.n_epochs}")
print(f"Trainset: {trainset.n_users} users, {trainset.n_items} items, {trainset.n_ratings} ratings")


Beste RMSE (CV): 1.6181
Beste parameters: {'n_factors': 50, 'n_epochs': 20, 'lr_all': 0.01, 'reg_all': 0.1}

Validatie RMSE (SVD op val_df): 1.5604

Final SVD model getraind op train+val — factors=50, epochs=20
Trainset: 23802 users, 15749 items, 465575 ratings


## Hybride Scoring & Cold-Start Afhandeling

De collaborative score komt van het SVD-model inclusief de biases. De content score (cosine similarity, 0–1) wordt geschaald naar 0–10 zodat beide onderdelen vergelijkbaar zijn.

**Cold-start regels:**
- Gebruikers met < 5 training ratings → popularity fallback met genrediversiteit.
- Films met < 5 training ratings → gebruikersgemiddelde als collaborative score, zodat het content-signaal domineert.

In [12]:
# gewichten
COLLAB_WEIGHT = 0.8
CONTENT_WEIGHT = 0.2

# Handige lookups
user_mean_rating = train_val_df.groupby('user_id')['rating'].mean().to_dict()
ratings_per_movie = train_val_df.groupby('movie_id').size().to_dict()
watched_movies = train_val_df.groupby('user_id')['movie_id'].apply(set).to_dict()
all_movies = set(items['movie_id'].unique())
global_avg = train_val_df['rating'].mean()

def predict_svd_scores(user_id, movie_ids):
    """Bereken SVD-voorspellingen voor een lijst films."""
    n = len(movie_ids)
    fallback = user_mean_rating.get(user_id, global_avg)

    try:
        inner_uid = svd.trainset.to_inner_uid(user_id)
    except ValueError:
        return np.full(n, fallback)

    pu = svd.pu[inner_uid]
    bu = svd.bu[inner_uid]
    mu = svd.trainset.global_mean

    predictions = np.full(n, fallback)
    valid = np.zeros(n, dtype=bool)
    inner_iids = np.zeros(n, dtype=int)

    for i, mid in enumerate(movie_ids):
        try:
            inner_iids[i] = svd.trainset.to_inner_iid(mid)
            valid[i] = True
        except ValueError:
            pass

    if valid.any():
        qi_batch = svd.qi[inner_iids[valid]]
        bi_batch = svd.bi[inner_iids[valid]]
        predictions[valid] = mu + bu + bi_batch + qi_batch @ pu

    return np.clip(predictions, 0, 10)

print(f"Hybride gewichten: collab={COLLAB_WEIGHT}, content={CONTENT_WEIGHT}")
print(f"Cold-start drempel: gebruiker < {COLD_THRESHOLD} ratings → popularity fallback")
print(f"Cold-start film drempel: < 5 ratings → user mean ({global_avg:.2f} global avg)")

Hybride gewichten: collab=0.8, content=0.2
Cold-start drempel: gebruiker < 5 ratings → popularity fallback
Cold-start film drempel: < 5 ratings → user mean (7.33 global avg)


## Aanbevelingsfunctie met Diversiteit

Per gebruiker:
- Bereken de hybride score voor alle onbeoordeelde films.
- Selecteer de top 2 op score.
- Selecteer een diverse 3e film waarvan de genres niet een subset zijn van de eerste twee voor expliciete serendipity.

Cold-start gebruikers (< 5 training ratings) ontvangen populariteitsgebaseerde keuzes met afgedwongen genrediversiteit zodat er sneller over de gebruiker geleerd kan worden.

## Helpe functions

In [13]:
def recommend(user_id, n=3):
    """
    Geeft een DataFrame met de top-n aanbevolen films voor een gebruiker.
   Expliciete genre diverwsiteit voor de 3de keuze
    """
    rated = watched_movies.get(user_id, set())
    candidates = np.array(list(all_movies - rated))
    if len(candidates) == 0:
        return pd.DataFrame(columns=['rank', 'movie_id', 'title', 'genres', 'score'])

    is_cold = user_id in cold_users

    # Bereken scores voor alle kandidaten
    if is_cold:
        scores = np.array([popularity_dict.get(m, 0.0) for m in candidates])
    else:
        collab_scores = predict_svd_scores(user_id, candidates)
        ui = user_to_index.get(user_id)
        if ui is not None:
            cb_scores = np.array([
                content_similarity[ui, movie_to_index[m]] * 10.0
                if m in movie_to_index else 0.0
                for m in candidates
            ])
        else:
            cb_scores = np.zeros(len(candidates))
        scores = COLLAB_WEIGHT * collab_scores + CONTENT_WEIGHT * cb_scores

    # Sorteer op score (hoog → laag)
    order = np.argsort(-scores)
    sorted_candidates = candidates[order]
    sorted_scores = scores[order]

    if len(sorted_candidates) <= n or n <= 2:
        return format_recommendations(list(zip(sorted_candidates[:n], sorted_scores[:n])))

    if is_cold:
        return diverse_popular_picks(sorted_candidates, sorted_scores, n)

    # Top 2 op basis van hybride score
    top2 = [(sorted_candidates[0], sorted_scores[0]),
            (sorted_candidates[1], sorted_scores[1])]

    # Genres die al in top 2 zitten
    top2_genres = set()
    for mid, _ in top2:
        if mid in movie_to_index:
            row = genre_matrix.loc[mid]
            top2_genres.update(row[row == 1].index)

    # Diverse 3e keuze: eerste kandidaat met genres die NIET een subset zijn van top-2
    diverse_pick = None
    for i in range(2, min(80, len(sorted_candidates))):
        mid = sorted_candidates[i]
        if mid in movie_to_index:
            movie_genres = set(genre_matrix.loc[mid][genre_matrix.loc[mid] == 1].index)
            if movie_genres and not movie_genres.issubset(top2_genres):
                diverse_pick = (mid, sorted_scores[i])
                break

    if diverse_pick is None:
        diverse_pick = (sorted_candidates[2], sorted_scores[2])

    return format_recommendations([top2[0], top2[1], diverse_pick])


def diverse_popular_picks(sorted_candidates, sorted_scores, n):
    """Kies top-n populaire films zonder genre-overlap tussen de keuzes."""
    picked, seen_genres = [], set()
    for mid, sc in zip(sorted_candidates, sorted_scores):
        if mid in movie_to_index:
            movie_genres = set(genre_matrix.loc[mid][genre_matrix.loc[mid] == 1].index)
            if movie_genres.isdisjoint(seen_genres) or len(picked) == 0:
                picked.append((mid, sc))
                seen_genres.update(movie_genres)
        if len(picked) >= n:
            break

    # Vul resterende plekken aan vanuit top als nodig
    if len(picked) < n:
        already_picked = {p[0] for p in picked}
        for mid, sc in zip(sorted_candidates, sorted_scores):
            if mid not in already_picked:
                picked.append((mid, sc))
            if len(picked) >= n:
                break
    return format_recommendations(picked[:n])


def format_recommendations(picked):
    """Zet de lijst van (movie_id, score) tuples om naar een -DataFrame."""
    rows = []
    for rank, (mid, sc) in enumerate(picked, 1):
        info = items[items['movie_id'] == mid]
        title = info['title'].values[0] if len(info) else f"Movie {mid}"
        genres = ', '.join(info['genres'].values[0]) if len(info) and isinstance(info['genres'].values[0], list) else ''
        rows.append({
            'rank': rank, 'movie_id': int(mid),
            'title': title, 'genres': genres,
            'score': round(float(sc), 3)
        })
    return pd.DataFrame(rows)

print("recommend() functie gereed.")

recommend() functie gereed.


In [14]:

def compute_rmse_hybrid(test_data):
    user_ids = test_data['user_id'].values
    movie_ids = test_data['movie_id'].values
    actuals = test_data['rating'].values
    preds = np.empty(len(actuals))

    mu = svd.trainset.global_mean
    for i in range(len(actuals)):
        uid, mid = user_ids[i], movie_ids[i]
        fallback = user_mean_rating.get(uid, global_avg)

        if uid in cold_users:
            preds[i] = movie_avg_dict.get(mid, global_avg)
            continue

        if ratings_per_movie.get(mid, 0) < 5:
            collab = fallback
        else:
            try:
                inner_uid = svd.trainset.to_inner_uid(uid)
                inner_iid = svd.trainset.to_inner_iid(mid)
                collab = np.clip(
                    mu + svd.bu[inner_uid] + svd.bi[inner_iid] + np.dot(svd.pu[inner_uid], svd.qi[inner_iid]),
                    0, 10
                )
            except ValueError:
                collab = fallback

        cb = get_content_score(uid, mid) * 10.0
        preds[i] = COLLAB_WEIGHT * collab + CONTENT_WEIGHT * cb

    return np.sqrt(mean_squared_error(actuals, np.clip(preds, 0, 10)))


def compute_rmse_baseline(test_data):
    actuals = test_data['rating'].values
    preds = np.array([movie_avg_dict.get(m, global_avg) for m in test_data['movie_id'].values])
    return np.sqrt(mean_squared_error(actuals, preds))


def precision_at_k(test_data, rec_func, k=3, threshold=7):
    precisions = []
    for uid in test_data['user_id'].unique():
        relevant = set(test_data[(test_data['user_id'] == uid) & (test_data['rating'] >= threshold)]['movie_id'])
        if not relevant:
            continue
        recs = rec_func(uid, n=k)
        if recs.empty:
            precisions.append(0.0)
            continue
        hits = len(set(recs['movie_id']) & relevant)
        precisions.append(hits / k)
    return np.mean(precisions) if precisions else 0.0


def baseline_recommend(user_id, n=3):
    rated = watched_movies.get(user_id, set())
    top = movie_stats[~movie_stats['movie_id'].isin(rated)].head(n)
    rows = []
    for rank, (_, r) in enumerate(top.iterrows(), 1):
        g = ', '.join(r['genres']) if isinstance(r['genres'], list) else str(r['genres'])
        rows.append({'rank': rank, 'movie_id': r['movie_id'],
                     'title': r['title'], 'genres': g,
                     'score': round(r['popularity_score'], 3)})
    return pd.DataFrame(rows)



## Evaluatie

Het hybride model en de popularity baseline worden uitsluitend geëvalueerd op de testset (data die niet is gebruikt voor training of hyperparametertuning). RMSE en Precision@3 worden berekend om de finale prestaties te meten.

In [15]:
# instelbare sample size (None = alle gebruikers)
user_subset = None

rmse_hybrid = compute_rmse_hybrid(test_df)
rmse_baseline = compute_rmse_baseline(test_df)

unique_users = test_df['user_id'].unique()

if user_subset is None:
    sample_users = unique_users
    sample_size = len(unique_users)
else:
    sample_size = min(user_subset, len(unique_users))
    sample_users = np.random.choice(unique_users, size=sample_size, replace=False)

sample_test_df = test_df[test_df['user_id'].isin(sample_users)]

print(f"\nPrecision@3 berekenen op {sample_size} testgebruiker{'s' if sample_size != 1 else ''}")

p3_hybrid = precision_at_k(sample_test_df, recommend, k=3, threshold=7)
p3_baseline = precision_at_k(sample_test_df, baseline_recommend, k=3, threshold=7)

# Resultaten
print(f"\n{'Metric':<20} {'Hybrid':>10} {'Baseline':>10}")
print(f"{'RMSE':<20} {rmse_hybrid:>10.4f} {rmse_baseline:>10.4f}")
print(f"{'Precision@3':<20} {p3_hybrid:>10.4f} {p3_baseline:>10.4f}")

print(f"\nValidatie RMSE (SVD): {val_rmse:.4f}")


Precision@3 berekenen op 19185 testgebruikers

Metric                   Hybrid   Baseline
RMSE                     1.5350     1.6009
Precision@3              0.0200     0.0399

Validatie RMSE (SVD): 1.5604


## Voorbeeldaanbevelingen

Top-3 aanbevelingen voor enkele gebruikers met verschillende activiteitsniveaus, inclusief de hybride score en genres.

In [16]:
# Kies 5 gebruikers met verschillende activiteitsniveaus
active_users = train_val_df['user_id'].value_counts()
example_users = list(active_users.index[:3])          # 3 meest actieve users
least_active = active_users.tail(2).index.tolist()    # 2 minst actieve users
example_users.extend(least_active)

for uid in example_users:
    n_rated = ratings_per_user.get(uid, 0)
    tag = "  ← cold-start (popularity)" if uid in cold_users else ""
    print(f"\n{'='*72}")
    print(f"  User {uid}   ({n_rated} training ratings){tag}")
    print(f"{'='*72}")
    recs = recommend(uid, n=3)
    for _, row in recs.iterrows():
        print(f"  #{int(row['rank'])}  {row['title']:<50s}  score={row['score']:.3f}")
        print(f"      Genres: {row['genres']}")


  User 17405   (5 training ratings)
  #1  The Dark Knight                                     score=8.395
      Genres: Action, Crime, Drama, Thriller
  #2  The Godfather: Part II                              score=8.337
      Genres: Crime, Drama
  #3  Oldeuboi                                            score=8.152
      Genres: Action, Drama, Mystery, Thriller

  User 26962   (5 training ratings)
  #1  The Dark Knight                                     score=9.055
      Genres: Action, Crime, Drama, Thriller
  #2  Fight Club                                          score=8.833
      Genres: Drama
  #3  Jaws                                                score=8.648
      Genres: Adventure, Drama, Thriller

  User 40861   (5 training ratings)
  #1  Joker                                               score=8.647
      Genres: Crime, Drama, Thriller
  #2  The Dark Knight                                     score=8.580
      Genres: Action, Crime, Drama, Thriller
  #3  Witness for the 

## Conclusie

Van de tien vooraf gedefinieerde requirements worden er acht compleet gehaald. Het systeem genereert drie unieke aanbevelingen per gebruiker, combineert collaborative en content-based filtering, gebruikt SVD met biasmodellering en bouwt gebruikersprofielen op basis van hoog gewaardeerde films. Daarnaast wordt een hybride scoringsfunctie toegepast en is een cold-start strategie geïmplementeerd. Om variatie te stimuleren wordt de derde aanbeveling bewust gekozen uit een ander genrecluster, waardoor diversiteit en serendipity worden vergroot.

### Niet volledig behaalde requirements

Twee requirements worden niet volledig gehaald.

- **Requirement: het hybride model moest beter presteren dan de populariteitsbaseline op alle evaluatiemetrics.**
  Het hybride model behaalt een lagere RMSE dan de baseline (1.5350 vs. 1.6009) en voorspelt daarmee individuele ratings nauwkeuriger. De*Precision@3 is echter lager (0.0200 vs. 0.0399). Dit komt deels doordat de dataset een relatief hoge gemiddelde rating heeft (7.33). Populaire films worden daardoor door veel gebruikers als relevant beschouwd, waardoor een eenvoudige populariteitsbaseline relatief vaak een een goede recommendation doet met de top-3 aanbevelingen.

- **Requirement: hoge rankingkwaliteit van aanbevelingen.**
  De ankingmetric Precision@3 is  laag. Dit komt deels doordat de testset per gebruiker slechts een klein aantal ratings bevat. Relevante films die het model aanbeveelt maar die niet in de testdata voorkomen worden in offline evaluatie als fout geteld, terwijl ze in werkelijkheid mogelijk wel relevant zijn voor de gebruiker.

### Keuze van evaluatiemetric

De keuze voor Precision@3 als een van mijn twee primaire rankingmetric blijkt achteraf minder geschikt voor deze implementatie. In het aanbevelingsalgoritme wordt namelijk expliciet diversiteit afgedwongen door de derde aanbeveling uit een ander genrecluster te selecteren. Deze ontwerpkeuze zorgt voor extra variatie en vergroot de kans op serendipity, maar het betekent ook dat de derde film niet altijd de hoogste scorende kandidaat is volgens het model.

### Interpretatie

Ondanks de lagere Precision@3 laat het model zien dat een hybride aanpak betere ratingvoorspellingen kan leveren dan een eenvoudige populariteitsbaseline. Tegelijkertijd zorgt de diversiteitsregel ervoor dat gebruikers sneller films uit verschillende genres te zien krijgen/

In een online systeem kan deze strategie voordelen hebben. De extra variatie in aanbevelingen kan helpen om nieuwe gebruikersvoorkeuren sneller te leren, waardoor een potentieel cold-start probleem sneller wordt opgelost zodra gebruikers beginnen met interacties.

### Mogelijke verbeteringen

Voor toekomstig werk zijn verschillende verbeteringen mogelijk:

* **Rijkere content features:** Naast genres ook acteurs, regisseurs of plotbeschrijvingen toevoegen zodat films inhoudelijk beter van elkaar te onderscheiden zijn.
* **Dynamische hybride gewichten:** De verhouding tussen collaborative en content scores automatisch leren in plaats van vaste gewichten te gebruiken.
* **Alternatieve evaluatiemetrics:** Metrics zoals **NDCG, coverage of diversiteitsmetrics** kunnen een completer beeld geven van de kwaliteit van aanbevelingen.
* **Meer trainingsdata per gebruiker:** Een hoger given-n of meer interactiedata kan helpen om betere latente gebruikers- en itemfactoren te leren.


## Bronnen

[1] K. Falk, Practical Recommender Systems. Manning Publications, 2019. https://github.com/Rishabh-creator601/Books/blob/master/ML-DL-BROAD/Practical%20Recommender%20Systems%20-%20Kim%20Falk%20(Manning,%202019).pdf

[2] Y. Koren, R. Bell, and C. Volinsky, “Matrix Factorization Techniques for Recommender Systems,” *Computer*, vol. 42, no. 8, pp. 30–37, Aug. 2009, doi: 10.1109/MC.2009.263

[3] G. Adomavicius and A. Tuzhilin, “Toward the Next Generation of Recommender Systems: A Survey of the State-of-the-Art and Possible Extensions,” IEEE Transactions on Knowledge and Data Engineering, vol. 17, no. 6, pp. 734–749, 2005. doi: 10.1109/TKDE.2005.99.

[4] Shani, G., Gunawardana, A. (2011). Evaluating Recommendation Systems. In: Ricci, F., Rokach, L., Shapira, B., Kantor, P. (eds) Recommender Systems Handbook. Springer, Boston, MA. https://doi.org/10.1007/978-0-387-85820-3_8

[5] Yuying Zhao, Yu Wang, Yunchao Liu, Xueqi Cheng, Charu C. Aggarwal, and Tyler Derr. 2025. Fairness and Diversity in Recommender Systems: A Survey. ACM Trans. Intell. Syst. Technol. 16, 1, Article 2 (February 2025), 28 pages. https://doi.org/10.1145/3664928